## OPTION-IMPLIED PROBABILITIES

The theory behind Breeden and Litzenberger (1978): call prices across strikes
pin down the risk-neutral distribution of the underlying at expiry. The second
strike derivative of the call price is the discounted density, the first is
the probability of finishing above the strike, and a butterfly spread is the
traded version of both. Checked numerically under Black-Scholes and under a
bimodal event distribution, with a look at why the raw recipe breaks on real
quotes.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# ----------------------------------------------------------------------------
# Imports
# ----------------------------------------------------------------------------

# Third-party imports
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import brentq
from scipy.stats import norm

## From state prices to call prices

Let $q(S)$ be the risk-neutral density of the underlying at expiry $T$. Any
European payoff $g(S_T)$ is worth its discounted risk-neutral expectation, so a
call struck at $K$ is

$$C(K) = e^{-rT}\int_K^\infty (S - K)\,q(S)\,dS.$$

Differentiating once in $K$ (Leibniz rule; the boundary term vanishes because
the integrand is zero at $S = K$),

$$\frac{\partial C}{\partial K} = -e^{-rT}\int_K^\infty q(S)\,dS = -e^{-rT}\,\mathbb{Q}(S_T > K),$$

and once more,

$$\frac{\partial^2 C}{\partial K^2} = e^{-rT}\,q(K).$$

So the slope of the call curve is a discounted digital option, and its
curvature is a discounted density: the price today of \$1 paid if $S_T$ lands
in $[K, K + dK]$, per unit $dK$. These are the Arrow-Debreu state prices. No
model for the dynamics of $S$ is needed; only that calls exist at every strike.
The same holds for puts, with $\partial P/\partial K = e^{-rT}\,\mathbb{Q}(S_T < K)$
and the same second derivative, by put-call parity.

Two consequences follow for free. Because $q \ge 0$, any arbitrage-free call
curve is convex in $K$, and because $\mathbb{Q}(S_T > K) \in [0, 1]$, its slope
lies in $[-e^{-rT}, 0]$.

## Inputs

A \$100 stock, three months to expiry, a 4% rate and 20% volatility for the
Black-Scholes benchmark.

In [3]:
spot = 100.0
r = 0.04  # continuously compounded rate
T = 0.25  # years
sigma = 0.20  # Black-Scholes volatility
forward = spot * np.exp(r * T)

K = np.linspace(50.0, 160.0, 2201)  # fine strike grid, 0.05 apart
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

In [4]:
def black_call(F, K, T, r, sigma):
    """Black-Scholes call on a forward F, discounted at r."""
    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return np.exp(-r * T) * (F * norm.cdf(d1) - K * norm.cdf(d2))


def lognormal_density(F, K, T, sigma):
    """Risk-neutral density of S_T under Black-Scholes, evaluated at K."""
    d2 = (np.log(F / K) - 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    return norm.pdf(d2) / (K * sigma * np.sqrt(T))


def butterfly_density(C, h, T, r):
    """Density from second differences of calls spaced h apart (interior strikes)."""
    return np.exp(r * T) * (C[:-2] - 2 * C[1:-1] + C[2:]) / h**2

## The butterfly spread

The second difference

$$\frac{C(K-h) - 2C(K) + C(K+h)}{h^2} \;\longrightarrow\; \frac{\partial^2 C}{\partial K^2} \quad (h \to 0)$$

is the price of a real portfolio: long one call at $K-h$, short two at $K$,
long one at $K+h$, all scaled by $1/h^2$. Its payoff is a triangle of height
$1/h$ and area 1 centred on $K$. As $h$ shrinks the triangle narrows into a
spike, the payoff of an Arrow-Debreu security that pays only if $S_T = K$. The
butterfly is how Breeden and Litzenberger make state prices tradable.

In [5]:
S_T = np.linspace(80.0, 120.0, 2001)
fig = go.Figure()
for color, h in zip(colors, [10.0, 5.0, 2.0]):
    payoff = (
        np.maximum(S_T - (100 - h), 0)
        - 2 * np.maximum(S_T - 100, 0)
        + np.maximum(S_T - (100 + h), 0)
    ) / h**2
    fig.add_trace(
        go.Scatter(
            x=S_T,
            y=payoff,
            mode="lines",
            name=f"h = {h:g}",
            line={"width": 2, "color": color},
        )
    )
fig.update_layout(
    title="Butterfly Payoff Scaled by 1/h², Centred on K = 100",
    xaxis_title="S_T",
    yaxis_title="Payoff",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.write_image("output/butterfly_payoff.png", scale=2)
fig.show()

## Check under Black-Scholes

With lognormal $S_T$ the density is known in closed form,
$q(K) = n(d_2)/(K\sigma\sqrt{T})$, so the butterfly recipe can be checked
against it. The central difference has error $\tfrac{h^2}{12}C''''(K)$, so the
worst-case gap should fall about fourfold each time $h$ halves.

In [6]:
exact = lognormal_density(forward, K, T, sigma)
for h in [4.0, 2.0, 1.0, 0.5]:
    strikes = np.arange(50.0, 160.0 + h / 2, h)
    recovered = butterfly_density(black_call(forward, strikes, T, r, sigma), h, T, r)
    gap = np.max(
        np.abs(recovered - lognormal_density(forward, strikes[1:-1], T, sigma))
    )
    print(f"h = {h:3.1f}: max density error {gap:.2e}")

C = black_call(forward, K, T, r, sigma)
h = K[1] - K[0]
recovered = butterfly_density(C, h, T, r)
print(f"total probability: {np.sum(recovered) * h:.6f}")
print(f"mean of S_T: {np.sum(K[1:-1] * recovered) * h:.4f} (forward {forward:.4f})")

h = 4.0: max density error 5.34e-04
h = 2.0: max density error 1.35e-04
h = 1.0: max density error 3.39e-05
h = 0.5: max density error 8.52e-06
total probability: 0.999998
mean of S_T: 101.0047 (forward 101.0050)


In [7]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=K,
        y=exact,
        mode="lines",
        name="Lognormal density",
        line={"width": 4, "color": colors[0]},
    )
)
fig.add_trace(
    go.Scatter(
        x=K[1:-1],
        y=recovered,
        mode="lines",
        name="e^{rT} ∂²C/∂K²",
        line={"width": 2, "color": colors[1], "dash": "dash"},
    )
)
fig.update_layout(
    title="Density Recovered from Black-Scholes Call Prices",
    xaxis_title="S_T",
    yaxis_title="Density",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.write_image("output/black_scholes_density.png", scale=2)
fig.show()

## Probabilities of events

The first derivative gives event probabilities directly, with no density or
integration needed:

$$\mathbb{Q}(S_T < K) = 1 + e^{rT}\frac{\partial C}{\partial K}.$$

Under Black-Scholes $-e^{rT}\,\partial C/\partial K = N(d_2)$, so the
probability of finishing above $K$ is $N(d_2)$, not the call's delta $N(d_1)$.
The traded version is a tight call spread: long $K - h/2$, short $K + h/2$,
scaled by $1/h$, which approximates a digital paying \$1 above $K$.

In [8]:
for level in [80.0, 90.0, 110.0]:
    spread = (
        black_call(forward, level - 0.05, T, r, sigma)
        - black_call(forward, level + 0.05, T, r, sigma)
    ) / 0.1
    above = np.exp(r * T) * spread
    d2 = (np.log(forward / level) - 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    print(
        f"Q(S_T > {level:.0f}): call spread {above:.4f}, N(d2) {norm.cdf(d2):.4f}, "
        f"Q(S_T < {level:.0f}) {1 - above:.4f}"
    )

Q(S_T > 80): call spread 0.9887, N(d2) 0.9887, Q(S_T < 80) 0.0113
Q(S_T > 90): call spread 0.8651, N(d2) 0.8651, Q(S_T < 90) 0.1349
Q(S_T > 110): call spread 0.1832, N(d2) 0.1832, Q(S_T < 110) 0.8168


## Beyond lognormal: a scheduled binary event

The result is model-free, so it should recover any distribution. Take a stock
facing a binary event before expiry (an FDA decision, say): with probability
$w = 0.6$ it trades around 8% above the forward, otherwise around 12% below,
with 12% volatility in each scenario. The two means are set so the mixture's
mean is the forward, as risk-neutral pricing requires. A mixture of lognormals
prices calls as the same mixture of Black-Scholes prices,

$$C(K) = w\,C_{\text{BS}}(F_1, K, s_1) + (1 - w)\,C_{\text{BS}}(F_2, K, s_2),$$

so the true density is known and the recovered one can be checked.

In [9]:
w = 0.6  # probability of the good outcome
F_up = forward * 1.08
F_down = (forward - w * F_up) / (1 - w)  # forces w F_up + (1 - w) F_down = forward
s = 0.12  # volatility within each scenario


def mixture_call(K):
    return w * black_call(F_up, K, T, r, s) + (1 - w) * black_call(F_down, K, T, r, s)


def mixture_density(K):
    return w * lognormal_density(F_up, K, T, s) + (1 - w) * lognormal_density(
        F_down, K, T, s
    )


C_mix = mixture_call(K)
recovered_mix = butterfly_density(C_mix, h, T, r)
print(f"F_up: {F_up:.2f}, F_down: {F_down:.2f}")
print(
    f"max density error: {np.max(np.abs(recovered_mix - mixture_density(K[1:-1]))):.2e}"
)

F_up: 109.09, F_down: 88.88
max density error: 2.17e-07


A trader quoting these options in Black-Scholes terms maps each price to its
own implied volatility, and the curve is an inverted smile peaking between the two
outcomes. The two-humped density hides behind a
single curve of volatilities, and the recipe recovers it exactly from prices
without knowing where it came from.

In [10]:
def implied_vol(price, F, K, T, r):
    """Black-Scholes volatility that reproduces a call price."""
    return brentq(lambda vol: black_call(F, K, T, r, vol) - price, 1e-4, 5.0)


smile_K = np.arange(70.0, 131.0, 1.0)
smile = np.array([implied_vol(mixture_call(k), forward, k, T, r) for k in smile_K])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=smile_K,
        y=100 * smile,
        mode="lines",
        name="Implied vol",
        line={"width": 2, "color": colors[0]},
    )
)
fig.update_layout(
    title="Implied Volatility Smile of the Event Stock",
    xaxis_title="Strike",
    yaxis_title="Implied volatility (%)",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
)
fig.write_image("output/event_smile.png", scale=2)
fig.show()

In [11]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=K,
        y=mixture_density(K),
        mode="lines",
        name="True density",
        line={"width": 4, "color": colors[0]},
    )
)
fig.add_trace(
    go.Scatter(
        x=K[1:-1],
        y=recovered_mix,
        mode="lines",
        name="e^{rT} ∂²C/∂K²",
        line={"width": 2, "color": colors[1], "dash": "dash"},
    )
)
fig.add_trace(
    go.Scatter(
        x=K,
        y=lognormal_density(
            forward, K, T, implied_vol(mixture_call(forward), forward, forward, T, r)
        ),
        mode="lines",
        name="Lognormal at ATM vol",
        line={"width": 2, "color": colors[2], "dash": "dot"},
    )
)
fig.update_layout(
    title="Density Recovered from the Event Stock's Call Prices",
    xaxis_title="S_T",
    yaxis_title="Density",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.write_image("output/event_density.png", scale=2)
fig.show()

## The smile changes event probabilities

A tempting shortcut is to read $\mathbb{Q}(S_T > K)$ as $N(d_2)$ using the
implied volatility at $K$. With a smile that is wrong. Writing
$C(K) = C_{\text{BS}}(K, \sigma(K))$ and differentiating,

$$\mathbb{Q}(S_T > K) = N\big(d_2(\sigma(K))\big) - e^{rT}\,\mathcal{V}(K)\,\sigma'(K),$$

where $\mathcal{V}$ is the Black-Scholes vega. The slope of the smile moves
probability around, and the correction can be large where the smile is steep.

In [12]:
for level in [85.0, 95.0, 105.0, 115.0]:
    true_above = w * norm.cdf(
        (np.log(F_up / level) - 0.5 * s**2 * T) / (s * np.sqrt(T))
    ) + (1 - w) * norm.cdf((np.log(F_down / level) - 0.5 * s**2 * T) / (s * np.sqrt(T)))
    vol = implied_vol(mixture_call(level), forward, level, T, r)
    d1 = (np.log(forward / level) + 0.5 * vol**2 * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    vega = np.exp(-r * T) * forward * norm.pdf(d1) * np.sqrt(T)
    slope = (
        implied_vol(mixture_call(level + 0.01), forward, level + 0.01, T, r)
        - implied_vol(mixture_call(level - 0.01), forward, level - 0.01, T, r)
    ) / 0.02
    corrected = norm.cdf(d2) - np.exp(r * T) * vega * slope
    print(
        f"Q(S_T > {level:.0f}): true {true_above:.4f}, N(d2) at smile vol "
        f"{norm.cdf(d2):.4f}, with skew term {corrected:.4f}"
    )

Q(S_T > 85): true 0.9050, N(d2) at smile vol 0.9313, with skew term 0.9050
Q(S_T > 95): true 0.6441, N(d2) at smile vol 0.6588, with skew term 0.6441
Q(S_T > 105): true 0.4377, N(d2) at smile vol 0.3482, with skew term 0.4377
Q(S_T > 115): true 0.1088, N(d2) at smile vol 0.0806, with skew term 0.1088


## Why the raw recipe fails on real quotes

Everything above used exact prices on a strike grid 5 cents apart. Real
options list at a handful of strikes, often \$1 to \$5 apart, and each quote
is only known to within its bid-ask spread. Both hurt the second difference:

- **Spacing** $h$ adds bias of order $h^2 C''''$, which smears sharp features
  such as the event's two humps.
- **Noise** $\varepsilon$ in each price becomes noise of order
  $\varepsilon/h^2$ in the density, because three noisy prices are combined
  and divided by $h^2$.

Shrinking $h$ reduces the first and inflates the second. Below, the event
stock's calls are quoted with two cents of random error, a small fraction of a
typical spread, at \$1 and \$5 spacing.

In [13]:
rng = np.random.default_rng(0)
noise = 0.02  # standard deviation of quote error, $

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=K,
        y=mixture_density(K),
        mode="lines",
        name="True density",
        line={"width": 4, "color": colors[0]},
    )
)
for color, spacing in zip(colors[1:], [1.0, 5.0]):
    strikes = np.arange(60.0, 140.0 + spacing / 2, spacing)
    quotes = mixture_call(strikes) + rng.normal(0.0, noise, strikes.size)
    density = butterfly_density(quotes, spacing, T, r)
    fig.add_trace(
        go.Scatter(
            x=strikes[1:-1],
            y=density,
            mode="lines+markers",
            name=f"Butterflies, ${spacing:g} strikes",
            line={"width": 2, "color": color},
        )
    )
    print(
        f"${spacing:g} strikes: noise scale {np.exp(r * T) * noise * np.sqrt(6) / spacing**2:.4f}, "
        f"negative densities at {np.sum(density < 0)} of {density.size} strikes"
    )
fig.add_hline(y=0, line={"width": 1, "color": "gray"})
fig.update_layout(
    title="Butterfly Densities from Noisy Quotes",
    xaxis_title="S_T",
    yaxis_title="Density",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.write_image("output/noisy_butterflies.png", scale=2)
fig.show()

$1 strikes: noise scale 0.0495, negative densities at 29 of 79 strikes
$5 strikes: noise scale 0.0020, negative densities at 2 of 15 strikes


Fine strikes give a density swamped by noise, with negative "probabilities";
coarse strikes are stable but blur the two outcomes together and say nothing
between strikes or beyond the last one. The theory is exact, but it needs a
smooth, arbitrage-free call curve at every strike, and markets supply a few
noisy points. The practical methods fill that gap: smoothing the implied
volatility curve before differentiating (Shimko 1993) or solving for the
smoothest density that prices the quotes within their spreads (Jackwerth and
Rubinstein 1996).

## References

- Breeden, D. T. and Litzenberger, R. H. (1978). Prices of state-contingent
  claims implicit in option prices. *Journal of Business*, 51(4), 621-651.
- Shimko, D. (1993). Bounds of probability. *Risk*, 6(4), 33-37.
- Jackwerth, J. C. and Rubinstein, M. (1996). Recovering probability
  distributions from option prices. *Journal of Finance*, 51(5), 1611-1631.